In [ ]:
import requests # Imports the requests library for making HTTP requests

from config import HF_API_KEY # Imports the Hugging Face API key from a 'config' module

MODEL_ID = "facebook/bart-large-mnli" # Defines the ID of the Hugging Face model to be used

API_URL = f"https://router.huggingface.co/hf-inference/models/{MODEL_ID}" # Constructs the API endpoint URL for the model

HEADERS = {"Authorization": f"Bearer {HF_API_KEY}"} # Sets up the authorization headers with the API key

TOPICS = ["Sports", "Technology", "Business", "Politics", "Health"] # Defines a list of candidate labels (topics) for classification

def ask_hf(headline: str): # Defines a function to send a headline to the Hugging Face API
    payload = {"inputs": headline, "parameters": {"candidate_labels": TOPICS}} # Creates the payload for the API request, including the headline and candidate topics
    r = requests.post(API_URL, headers=HEADERS, json=payload, timeout=30) # Sends a POST request to the API with the payload and headers, with a 30-second timeout
    if not r.ok: # Checks if the HTTP request was successful
        raise RuntimeError(f"HF error {r.status_code}: {r.text}") # Raises a RuntimeError if the request failed
    return r.json() # Returns the JSON response from the API (a list of labels with scores)

def best_topic(preds: list): # Defines a function to find the best topic from the API predictions
    best = max(preds, key=lambda x: x["score"]) # Finds the topic with the highest score
    return best["label"], best["score"] # Returns the label and score of the best topic

def bar(score: float) -> str: # Defines a function to create a text-based confidence bar
    pct = score * 100 # Converts the score to a percentage
    blocks = int(pct // 10) # Calculates the number of filled blocks for the bar
    return "█" * blocks + "░" * (10 - blocks) # Returns the confidence bar string

def show(headline: str, preds: list): # Defines a function to display the classification results
    top_label, top_score = best_topic(preds) # Gets the best topic and its score
    print("\n" + "=" * 60) # Prints a separator line
    print("????️ News Topic Classifier") # Prints the classifier title
    print("=" * 60) # Prints another separator line
    print("Headline:", headline) # Prints the input headline
    print(f"Best topic: {top_label}") # Prints the best predicted topic
    print(f"Confidence: {round(top_score*100,1)}% [{bar(top_score)}]") # Prints the confidence percentage and bar

    print("\nTop 3 guesses:") # Prints a header for top 3 guesses
    top3 = sorted(preds, key=lambda x: x["score"], reverse=True)[:3] # Sorts predictions by score and takes the top 3
    for i, p in enumerate(top3, start=1): # Iterates through the top 3 predictions
        print(f"{i}. {p['label']:<11} {round(p['score']*100,1)}% [{bar(p['score'])}]") # Prints each top guess with its label, confidence, and bar
    print("=" * 60) # Prints a final separator line

def main(): # Defines the main function to run the news topic classifier
    print("Welcome! Type a news headline and I'll guess the topic.") # Welcomes the user
    print("Topics:", ", ".join(TOPICS)) # Lists the available topics
    print("Type 'exit' to stop.\n") # Instructs the user how to exit

    while True: # Starts an infinite loop for continuous input
        headline = input("Headline: ").strip() # Prompts the user for a headline and removes leading/trailing whitespace
        if headline.lower() == "exit": # Checks if the user wants to exit
            print("Bye! Keep coding ????") # Prints a farewell message
            break # Exits the loop
        if not headline: # Checks if the headline input is empty
            print("Please type a headline (not empty).\n") # Asks the user for a non-empty headline
            continue # Continues to the next iteration of the loop
        try: # Starts a try-except block to handle potential errors
            preds = ask_hf(headline) # Calls the ask_hf function to get predictions
            if isinstance(preds, list) and preds and "label" in preds[0]: # Validates the structure of the predictions
                show(headline, preds) # Calls the show function to display the results
            else: # If predictions are not in the expected format
                print("Oops! Unexpected reply:", preds) # Prints an error message with the unexpected reply
        except Exception as e: # Catches any other exceptions that might occur
            print("\n⚠️ Oops! Something went wrong.") # Prints a generic error message
            print("Reason:", e) # Prints the reason for the exception
            print("Tip: Check HF_API_KEY + internet.\n") # Provides troubleshooting tips

if __name__ == "__main__": # Ensures that main() runs only when the script is executed directly
    main() # Calls the main function to start the application

In [ ]:
# classifier.py - Complete Program

import requests # Imports the requests library for making HTTP requests

from config import HF_API_KEY # Imports the Hugging Face API key from a 'config' module


# Setup

MODEL = "facebook/bart-large-mnli" # Defines the ID of the Hugging Face model to be used

API_URL = f"https://router.huggingface.co/hf-inference/models/{MODEL}" # Constructs the API endpoint URL for the model

HEADERS = {"Authorization": f"Bearer {HF_API_KEY}"} # Sets up the authorization headers with the API key

LABELS = ["Spam", "Safe"] # Defines a list of candidate labels for classification (Spam or Safe)


def classify_message(message): # Defines a function to send a message to the Hugging Face API for classification

    payload = {"inputs": message, "parameters": {"candidate_labels": LABELS}} # Creates the payload for the API request, including the message and candidate labels

    response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=30) # Sends a POST request to the API with the payload and headers, with a 30-second timeout

    if not response.ok: # Checks if the HTTP request was successful

        raise RuntimeError(f"API error: {response.status_code}") # Raises a RuntimeError if the request failed

    data = response.json() # Parses the JSON response from the API

    results = list(zip(data["labels"], data["scores"])) # Combines labels and scores into a list of tuples

    return sorted(results, key=lambda x: x[1], reverse=True) # Returns the results sorted by score in descending order


def show_results(message, results): # Defines a function to display the classification results to the user

    label, score = results[0] # Extracts the top predicted label and its score

    print("\n" + "=" * 44) # Prints a separator line

    print("Spam vs Safe Message Classifier") # Prints the classifier title

    print("=" * 44) # Prints another separator line

    print(f"Message: {message}") # Prints the input message

    print(f"Result: {label} ({score*100:.1f}%)\n") # Prints the top result with its confidence percentage

    for i, (lbl, scr) in enumerate(results, 1): # Iterates through all classification results

        print(f"{i}. {lbl}: {scr*100:.1f}%") # Prints each label and its score

    if label == "Spam": # Checks if the top label is "Spam"

        print("\n⚠️  Don't click links or share info!") # Prints a warning message for spam

    else: # If the top label is "Safe"

        print("\n✅ Looks safe, stay alert!") # Prints a reassuring message for safe messages

    print("=" * 44) # Prints a final separator line


def main(): # Defines the main function to run the spam classifier application

    print("Spam Classifier - Type 'exit' to quit\n") # Welcomes the user and provides exit instructions

    while True: # Starts an infinite loop for continuous message input

        msg = input("Message: ").strip() # Prompts the user for a message and removes leading/trailing whitespace

        if msg.lower() == "exit": # Checks if the user wants to exit

            print("Goodbye!") # Prints a farewell message

            break # Exits the loop

        if not msg: # Checks if the message input is empty

            print("Enter a message!\n") # Asks the user to enter a message

            continue # Continues to the next iteration of the loop

        try: # Starts a try-except block to handle potential errors

            results = classify_message(msg) # Calls the classify_message function to get predictions

            show_results(msg, results) # Calls the show_results function to display the classification results

        except Exception as e: # Catches any exceptions that might occur during the process

            print(f"\n❌ Error: {e}\n") # Prints an error message with the exception details


if __name__ == "__main__": # Ensures that main() runs only when the script is executed directly

    main() # Calls the main function to start the application